In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed

# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

In [ ]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.image as img

import cv2
import itertools
import pathlib
import warnings
from PIL import Image
from random import randint
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef as MCC
from sklearn.metrics import balanced_accuracy_score as BAS
from sklearn.metrics import classification_report, confusion_matrix


from tensorflow import keras
from keras import layers
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.preprocessing import image_dataset_from_directory
##from keras.utils.vis_utils import plot_model
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.layers import Conv2D, Flatten
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator as IDG
from tensorflow.keras.layers import SeparableConv2D, BatchNormalization, GlobalAveragePooling2D

from distutils.dir_util import copy_tree, remove_tree

import os
#print(os.listdir("../input/alzheimer-mri-dataset/Dataset"))

print("TensorFlow Version:", tf.__version__)

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
X_train =np.load('/input/updated-sars-covid-ct-scan-dataset/sars_covid_ct_scan dataset/X_train_sars_ct.npy')
y_train =np.load('/input/updated-sars-covid-ct-scan-dataset/sars_covid_ct_scan dataset/y_train_sars_ct.npy')

X_test =np.load('/input/updated-sars-covid-ct-scan-dataset/sars_covid_ct_scan dataset/X_test_sars_ct.npy')
y_test =np.load('/input/updated-sars-covid-ct-scan-dataset/sars_covid_ct_scan dataset/y_test_sars_ct.npy')

X_train.shape,y_train.shape, X_test.shape,y_test.shape

In [ ]:
X_train_c =np.load('/input/pulmonary-dataset/X_train_pulmonary_chest.npy')
y_train_c =np.load('/input/pulmonary-dataset/y_train_pulmonary_chest.npy')

X_test_c =np.load('/input/pulmonary-dataset/X_test_pulmonary_chest.npy')
y_test_c =np.load('/input/pulmonary-dataset/y_test_pulmonary_chest.npy')

X_train_c.shape,y_train_c.shape, X_test_c.shape,y_test_c.shape

In [ ]:
import numpy as np
import cv2

def rotate_image(image, angle):
    """
    Rotate the image by the specified angle.
    """
    center = tuple(np.array(image.shape[1::-1]) / 2)
    rotation_matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated_image = cv2.warpAffine(image, rotation_matrix, image.shape[1::-1], flags=cv2.INTER_LINEAR)
    return rotated_image

def translate_image(image, tx, ty):
    """
    Translate the image by the specified translation parameters.
    """
    translation_matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    translated_image = cv2.warpAffine(image, translation_matrix, image.shape[1::-1])
    return translated_image

# Example data
#X_train = np.random.rand(100, 28, 28)  # Assuming 100 images of size 28x28
#y_train = np.random.randint(0, 10, 100)  # Assuming 100 labels

# Augmentation parameters
rotation_angles = [20]
translations = [(5, 5)]

augmented_X_train = []
augmented_y_train = []

for image, label in zip(X_train_c, y_train_c):
    # Original image
    #augmented_X_train.append(image)
    #augmented_y_train.append(label)

    # Augment with rotations
    for angle in rotation_angles:
        rotated_image = rotate_image(image, angle)
        augmented_X_train.append(rotated_image)
        augmented_y_train.append(label)

    # Augment with translations
    for tx, ty in translations:
        translated_image = translate_image(image, tx, ty)
        augmented_X_train.append(translated_image)
        augmented_y_train.append(label)

# Convert lists to numpy arrays
augmented_X_train = np.array(augmented_X_train)
augmented_y_train = np.array(augmented_y_train)

# Shuffle the data
shuffle_indices = np.random.permutation(len(augmented_X_train))
augmented_X_train = augmented_X_train[shuffle_indices]
augmented_y_train = augmented_y_train[shuffle_indices]
augmented_X_train.shape, augmented_y_train.shape
# Now, augmented_X_train and augmented_y_train contain the augmented dataset.

In [ ]:
random_indices = np.random.choice(640, 64, replace=False)

X_train_c1 = X_train_c[random_indices]
y_train_c1 = y_train_c[random_indices]

X_train_c1.shape, y_train_c1.shape

In [ ]:
X_train_c = np.concatenate((X_train_c,X_train_c1, augmented_X_train), axis=0)
y_train_c = np.concatenate((y_train_c, y_train_c1, augmented_y_train), axis=0)
X_train_c.shape, y_train_c.shape

In [ ]:
X_train.shape,X_test.shape, y_train.shape,y_test.shape

In [ ]:
from tensorflow.keras.utils import to_categorical
y_train_c = to_categorical(y_train_c)
y_train = to_categorical(y_train)

y_test = to_categorical(y_test)
y_test_c = to_categorical(y_test_c)

y_train_c.shape, y_train.shape, y_test.shape, y_test_c.shape

In [ ]:
random_indices = np.random.choice(160, 17, replace=False)

X_test_c1 = X_test_c[random_indices]
y_test_c1 = y_test_c[random_indices]

X_test_c1.shape, y_test_c1.shape

In [ ]:
X_test_c = np.concatenate((X_test_c,X_test_c, X_test_c, X_test_c1), axis=0)
y_test_c = np.concatenate((y_test_c, y_test_c, y_test_c, y_test_c1), axis=0)
X_test_c.shape, y_test_c.shape

In [ ]:
import tensorflow as tf
from keras.datasets import mnist
import cv2
import os
import pathlib
from keras.layers import Conv2D, Conv2DTranspose, Dropout, Dense, Reshape, LayerNormalization, LeakyReLU
from keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import f1_score, recall_score, precision_score

In [ ]:
plt.figure(figsize = (12, 4))
for i in range(16):
    plt.subplot(2, 8, (i + 1))
    plt.imshow(X_train[i], cmap = 'gray')
    #plt.title(y_train[i])
plt.show()

In [ ]:
plt.figure(figsize = (12, 4))
for i in range(16):
    plt.subplot(2, 8, (i + 1))
    plt.imshow(X_train_c[i], cmap = 'gray')
    #plt.title(y_train[i])
plt.show()

In [ ]:
#images_train_br.shape,labels_train_br.shape,images_test_br.shape,labels_test_br.shape
#y_train_br.shape, y_test_br.shape, y_train_ct.shape, y_test_ct.shape

In [ ]:
##SCA
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
import tensorflow as tf

class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
        super(SpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='selu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer()
        self.channel_attention = ChannelAttentionLayer()
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=self.spatial_noise_factor)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=self.channel_noise_factor)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

##SCA
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
import tensorflow as tf

class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
        super(SpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer()
        self.channel_attention = ChannelAttentionLayer()
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=self.spatial_noise_factor)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=self.channel_noise_factor)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling2D, Dense

def spatial_attention_layer(inputs):
    attention_map = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')(inputs)
    return tf.multiply(inputs, attention_map)

def channel_attention_layer(inputs):
    avg_pool = GlobalAveragePooling2D()(inputs)
    dense1_out = Dense(units=inputs.shape[-1] // 2, activation='selu')(avg_pool)
    channel_attention_weights = Dense(units=inputs.shape[-1], activation='sigmoid')(dense1_out)
    channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
    return tf.multiply(inputs, channel_attention_weights)

def combined_attention_noise_layer(inputs, spatial_noise_factor=1.0, channel_noise_factor=1.0):
    spatial_attention_output = spatial_attention_layer(inputs)
    channel_attention_output = channel_attention_layer(inputs)

    # Add spatial attention noise
    spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                mean=0, stddev=spatial_noise_factor)

    # Add channel attention noise
    channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                mean=0, stddev=channel_noise_factor)

    # Clip attention maps to ensure they are within the valid range [0, 1]
    spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
    channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

    # Combine attention mechanisms
    combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
    return tf.multiply(inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor
        self.trainable_weight = self.add_weight(name='trainable_weight', shape=(1,), initializer='ones', trainable=True)

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=self.spatial_noise_factor)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=self.channel_noise_factor)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms with trainable weight
        combined_attention = self.trainable_weight * spatial_attention_output * channel_attention_output
        return tf.multiply(inputs, combined_attention)

'''# Example Usage:
model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.datasets import cifar10
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

# Define the Spatial Attention Layer
class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
        super(SpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

# Define the Channel Attention Layer
class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

# Define the Combined Attention Noise Layer
class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer()
        self.channel_attention = ChannelAttentionLayer()
        self.spatial_noise_factor = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_factor')
        self.channel_noise_factor = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_factor')

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, stddev=self.spatial_noise_factor)
        spatial_attention_output += spatial_noise

        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, stddev=self.channel_noise_factor)
        channel_attention_output += channel_noise

        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)



In [ ]:
import tensorflow as tf

class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')
        super(SpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer()
        self.channel_attention = ChannelAttentionLayer()
        
        # Trainable weights to modulate noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        # Noise factors
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        # Scale the provided noise factors with trainable weights
        spatial_noise_factor_scaled = self.spatial_noise_factor * self.spatial_noise_weight
        channel_noise_factor_scaled = self.channel_noise_factor * self.channel_noise_weight

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, stddev=spatial_noise_factor_scaled)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, stddev=channel_noise_factor_scaled)

        # Add noise to attention outputs
        spatial_attention_output += spatial_noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.datasets import cifar10
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

class SpatialAttentionLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, **kwargs):
        super(SpatialAttentionLayer, self).__init__(**kwargs)
        self.spatial_noise_factor = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_factor')
        self.convolution = tf.keras.layers.Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')

    def call(self, inputs):
        spatial_noise = tf.random.normal(shape=tf.shape(inputs), mean=0, stddev=self.spatial_noise_factor)
        inputs_with_noise = inputs + spatial_noise
        attention_weights = self.convolution(inputs_with_noise)
        return tf.multiply(inputs, attention_weights)

class ChannelAttentionLayer(tf.keras.layers.Layer):
    def __init__(self,channel_noise_factor=1.0, **kwargs):
        super(ChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = tf.keras.layers.GlobalAveragePooling2D()
        self.dense1 = tf.keras.layers.Dense(units=input_shape[-1] // 2, activation='relu')
        self.dense2 = tf.keras.layers.Dense(units=input_shape[-1], activation='sigmoid')
        super(ChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        
        # Reshape channel_attention_weights to match the shape of inputs
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        channel_attention_weights = tf.tile(channel_attention_weights, [1, inputs.shape[1], inputs.shape[2], 1])
        
        return tf.multiply(inputs, channel_attention_weights)
    
class CombinedAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(CombinedAttentionNoiseLayer, self).__init__(**kwargs)
        self.spatial_attention = SpatialAttentionLayer(spatial_noise_factor)
        self.channel_attention = ChannelAttentionLayer(channel_noise_factor)

    def call(self, inputs):
        spatial_attention_output = self.spatial_attention(inputs)
        channel_attention_output = self.channel_attention(inputs)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)



In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  #trainable=True
                                 )
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', 
                            #trainable=True
                           )
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', #trainable=True
                           )
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Scale the noise factors with trainable weights
        spatial_noise_factor_scaled = self.spatial_noise_factor * self.spatial_noise_weight
        channel_noise_factor_scaled = self.channel_noise_factor * self.channel_noise_weight
        
        #spatial_noise_factor_scaled = self.spatial_noise_factor - spatial_noise_factor_scaled
        #channel_noise_factor_scaled = self.channel_noise_factor - channel_noise_factor_scaled


        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=spatial_noise_factor_scaled)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=channel_noise_factor_scaled)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Use trainable weights directly without scaling
        spatial_noise_factor_scaled = self.spatial_noise_weight
        channel_noise_factor_scaled = self.channel_noise_weight

        # Add spatial attention noise
        spatial_attention_output += tf.random.normal(shape=tf.shape(spatial_attention_output),
                                                    mean=0, stddev=spatial_noise_factor_scaled)

        # Add channel attention noise
        channel_attention_output += tf.random.normal(shape=tf.shape(channel_attention_output),
                                                    mean=0, stddev=channel_noise_factor_scaled)

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, spatial_noise_factor_scaled)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, channel_noise_factor_scaled)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + spatial_noise
        channel_noise *= self.channel_noise_weight + channel_noise

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
## more correct protection
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
## more correct protection for mm
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense, Add

class TrainableSpatialAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer1, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        self.concat = Add()
        super(TrainableSpatialAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        inputs1, inputs2 = inputs
        attention_weights1 = self.convolution(inputs1)
        attention_weights2 = self.convolution(inputs2)
        attention_weights = self.concat([attention_weights1, attention_weights2])
        con_inputs = self.concat([inputs1, inputs2])
        return tf.multiply(con_inputs, attention_weights)

class TrainableChannelAttentionLayer1(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer1, self).__init__(**kwargs)

    def build(self, input_shape):
        inputs1, inputs2 = input_shape
        num_channels = inputs1[-1]
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.concat = Add()
        self.dense1 = Dense(units=inputs1[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=inputs1[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer1, self).build(input_shape)

    def call(self, inputs):
        inputs1, inputs2 = inputs
        avg_pool1 = self.global_avg_pooling(inputs1)
        avg_pool2 = self.global_avg_pooling(inputs2)
        avg_pool = self.concat([avg_pool1, avg_pool2])
        
        dense1_out = self.dense1(avg_pool)
        
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)

        con_inputs = self.concat([inputs1, inputs2])
        #return tf.multiply(coninputs, attention_weights)

        return tf.multiply(con_inputs, channel_attention_weights)

'''class TrainableCombinedAttentionLayer1(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer1, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        inputs1, inputs2 = inputs
        spatial_attention_output = inputs1
        channel_attention_output = inputs1

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs1, combined_attention)
'''

class TrainableCombinedAttentionLayer1(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, **kwargs):
        super(TrainableCombinedAttentionLayer1, self).__init__(**kwargs)
        
        # Single attention layer for both spatial and channel attention
        self.spatial_attention = TrainableSpatialAttentionLayer1()
        self.channel_attention = TrainableChannelAttentionLayer1()
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor
        self.concat = Add()

    def call(self, inputs):
        inputs1, inputs2 = inputs
        
        # Apply spatial attention
        spatial_attention_output = self.spatial_attention([inputs1, inputs2])
        
        # Apply channel attention
        channel_attention_output = self.channel_attention([inputs1, inputs2])

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), 
                                         mean=0, stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), 
                                         mean=0, stddev=self.channel_noise_factor)
        
        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Scale the noise with trainable weights
        #spatial_noise *= self.spatial_noise_weight
        #channel_noise *= self.channel_noise_weight

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        '''# Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention = tf.multiply(spatial_attention_output, channel_attention_output)
        '''
        
        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        con_inputs = self.concat([inputs1, inputs2])
        
        return tf.multiply(con_inputs, combined_attention)


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model


def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(x1, x2):
    #x1, x2 = x
    
    print('x1:',x1.shape)
    print('x2:',x2.shape)
    
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = Activation('relu')(x1)
    
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)

    # Stack of residual blocks
    
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x2)
    x2 = BatchNormalization()(x2)
    x2 = Activation('relu')(x2)
    
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)
    
    print('x1:',x1.shape)
    print('x2:',x2.shape)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    x1 = residual_block(x, filters=64)
    
    x1 = residual_block(x1, filters=64)
    
    
    x2 = residual_block(x, filters=64)
    
    x2 = residual_block(x2, filters=64)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x1 = residual_block(x1, filters=128)
    
    x2 = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x2 = residual_block(x2, filters=128)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x1 = residual_block(x1, filters=256)
    
    x2 = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x2 = residual_block(x2, filters=256)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x1 = residual_block(x1, filters=512)
    
    
    x2 = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x2 = residual_block(x2, filters=512)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    return x

'''input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)



#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer
inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs2)
    
x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)

con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

x = GlobalAveragePooling2D()(con)
print('GlobalAveragePooling2D x:',x.shape)

outputs1 = Dense(5, activation='softmax')(x)
outputs2 = Dense(7, activation='softmax')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])
#return model
#print(model.summary())
'''
'''def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    
    inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
    inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs2)
    
    
    x = branch_ResNet18(inputs_layer1, inputs_layer2)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, 
                                        channel_noise_factor=1.0,num_layers=1)([x, x])
    
    
    out = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    
    outputs1 = Dense(2, activation='sigmoid')(out)
    
    outputs2 = Dense(2, activation='sigmoid')(out)
    


    # Create the model
    model = Model([inputs1, inputs2], 
                  [outputs1, outputs2])
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()
'''

input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)



#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer

#x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)

#con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])
inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                 channel_noise_factor=1.0)(inputs2)


con = branch_ResNet18(inputs_layer1, inputs_layer2)
    
x = GlobalAveragePooling2D()(con)
print('GlobalAveragePooling2D x:',x.shape)

outputs1 = Dense(2, activation='sigmoid')(x)
outputs2 = Dense(2, activation='sigmoid')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])



# Display the model summary
#resnet18.summary()


model.compile(optimizer='adam', loss=['binary_crossentropy', 
                                         'binary_crossentropy',#'categorical_crossentropy'
                                        ], metrics=['accuracy','accuracy', 
                                                    #'accuracy'
                                                   ])

#X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape
'''
print(images_train_br.shape, labels_train_br.shape)
print(images_train_ct.shape, labels_train_ct.shape)
'''

model.fit([X_train,
              images_train_br], 
             
             [labels_train_pneu,
              labels_train_br], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
             #verbose=0,
          validation_split=0.2)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model


def residual_block(x, filters, strides=(1, 1), use_projection=False):
    #x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
     #                                                   )([x, x])
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x, x])
    
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x, x])
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(x1, x2):
    #x1, x2 = x
    
    print('x1:',x1.shape)
    print('x2:',x2.shape)
    
    x1 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x1)
    x1 = BatchNormalization()(x1)
    x1 = Activation('relu')(x1)
    
    x1 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x1)

    # Stack of residual blocks
    
    x2 = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x2)
    x2 = BatchNormalization()(x2)
    x2 = Activation('relu')(x2)
    
    x2 = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x2)
    
    print('x1:',x1.shape)
    print('x2:',x2.shape)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    x1 = residual_block(x, filters=64)
    
    x1 = residual_block(x1, filters=64)
    
    
    x2 = residual_block(x, filters=64)
    
    x2 = residual_block(x2, filters=64)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x1 = residual_block(x1, filters=128)
    
    x2 = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x2 = residual_block(x2, filters=128)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x1 = residual_block(x1, filters=256)
    
    x2 = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x2 = residual_block(x2, filters=256)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    
    x1 = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x1 = residual_block(x1, filters=512)
    
    
    x2 = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x2 = residual_block(x2, filters=512)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, channel_noise_factor=1.0,
                                                        )([x1, x2])
    return x

'''input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)



#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer
inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs2)
    
x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)

con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])

x = GlobalAveragePooling2D()(con)
print('GlobalAveragePooling2D x:',x.shape)

outputs1 = Dense(5, activation='softmax')(x)
outputs2 = Dense(7, activation='softmax')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])
#return model
#print(model.summary())
'''
'''def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    
    inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
    inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs2)
    
    
    x = branch_ResNet18(inputs_layer1, inputs_layer2)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    
    x = TrainableCombinedAttentionLayer1(spatial_noise_factor=1.0, 
                                        channel_noise_factor=1.0,num_layers=1)([x, x])
    
    
    out = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    
    outputs1 = Dense(2, activation='sigmoid')(out)
    
    outputs2 = Dense(2, activation='sigmoid')(out)
    


    # Create the model
    model = Model([inputs1, inputs2], 
                  [outputs1, outputs2])
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()
'''

input_shape=(128, 128, 3)
inputs1 = Input(shape=input_shape)
inputs2 = Input(shape=input_shape)



#input_data = Input(shape=input_shape, name='input_data')
# Initial convolutional layer

#x1, x2 = residual_GLC_branch1(inputs1, inputs2)
#print('x:',x.shape)

#con = tf.keras.layers.Concatenate(axis=-1)([x1, x2])
inputs_layer1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0)(inputs1)
    
inputs_layer2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                 channel_noise_factor=1.0)(inputs2)


con = branch_ResNet18(inputs_layer1, inputs_layer2)
    
x = GlobalAveragePooling2D()(con)
print('GlobalAveragePooling2D x:',x.shape)

outputs1 = Dense(2, activation='sigmoid')(x)
outputs2 = Dense(2, activation='sigmoid')(x)

# Create the model
model = Model([inputs1, inputs2], [outputs1, outputs2])



# Display the model summary
#resnet18.summary()


model.compile(optimizer='adam', loss=['binary_crossentropy', 
                                         'binary_crossentropy',#'categorical_crossentropy'
                                        ], metrics=['accuracy','accuracy', 
                                                    #'accuracy'
                                                   ])

#X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape
'''
print(images_train_br.shape, labels_train_br.shape)
print(images_train_ct.shape, labels_train_ct.shape)
'''
#
model.fit([X_train,
              X_train_c], 
             [y_train,
              y_train_c], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
             #verbose=0,
          validation_split=0.2)

In [ ]:
model.fit([X_train,
              X_train_c], 
             [y_train,
              y_train_c], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
             #verbose=0,
          validation_split=0.2)

In [ ]:
'''
[X_train,
              X_train_c], 
             [y_train,
              y_train_c]
'''
model.evaluate([X_test,X_test_c], [y_test,y_test_c])

In [ ]:
y_test_c.shape,y_test.shape

In [ ]:
'''model.fit([X_train,
              images_train_br], 
             [labels_train_pneu,
              labels_train_br], 
             epochs=10, 
          validation_split=0.2)'''

In [ ]:
import tensorflow as tf

def pgd_attack(model, X_list, y_list, epsilon, alpha, num_iter, batch_size=32):
    """
    PGD adversarial attack on the model with multiple inputs.
    
    Parameters:
        model (tf.keras.Model): The target model to be attacked.
        X_list (list of tf.Tensor): List of input data tensors.
        y_list (list of tf.Tensor): List of true label tensors.
        epsilon (float): Perturbation size.
        alpha (float): Step size for PGD.
        num_iter (int): Number of iterations for PGD.
        batch_size (int): Batch size for processing inputs.
        
    Returns:
        adv_X_list (list of tf.Tensor): List of adversarial examples.
    """
    adv_X_list = [tf.identity(X) for X in X_list]  # Initialize adversarial examples with original inputs
    num_samples = X_list[0].shape[0]  # Assuming all inputs have the same number of samples
    
    for batch_start in range(0, num_samples, batch_size):
        batch_end = min(batch_start + batch_size, num_samples)
        batch_adv_X_list = [X[batch_start:batch_end] for X in adv_X_list]
        batch_y_list = [y[batch_start:batch_end] for y in y_list]
        
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                tape.watch(batch_adv_X_list)
                predictions = model(batch_adv_X_list)
                loss = sum([tf.keras.losses.binary_crossentropy(y, pred) for y, pred in zip(batch_y_list, predictions)])
            
            gradients = tape.gradient(loss, batch_adv_X_list)
            signed_grad = [tf.sign(grad) for grad in gradients]
            perturbations = [alpha * grad for grad in signed_grad]
            batch_adv_X_list = [tf.clip_by_value(X + perturbation, X - epsilon, X + epsilon) for X, perturbation in zip(batch_adv_X_list, perturbations)]
            batch_adv_X_list = [tf.clip_by_value(X, 0, 1) for X in batch_adv_X_list]  # Clip to valid image range [0, 1]
        
        # Update the adversarial examples back to the original list
        for i in range(len(adv_X_list)):
            indices = tf.range(batch_start, batch_end)
            adv_X_list[i] = tf.tensor_scatter_nd_update(adv_X_list[i], tf.expand_dims(indices, axis=1), batch_adv_X_list[i])
    
    return adv_X_list


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

#model = resnet18

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    '''
    [X_test,
              X_test_c], 
             [y_test,
              y_test_c]
              
              model.evaluate([X_test,X_test_c], [y_test,y_test_c])
    '''
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test,X_test_c], 
                          [y_test,y_test_c], epsilon, alpha, 
                                  num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(y_test_c, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

#model = resnet18

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 20

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    '''
    [X_test,
              X_test_c], 
             [y_test,
              y_test_c]
              
              model.evaluate([X_test,X_test_c], [y_test,y_test_c])
    '''
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test,X_test_c], 
                          [y_test,y_test_c], epsilon, alpha, 
                                  num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(y_test_c, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import tensorflow as tf

def mim_attack(model, X_list, y_list, epsilon, alpha, num_iter, decay_factor=1.0, momentum=0.9, batch_size=32):
    """
    Momentum Iterative Method (MIM) adversarial attack on the model with multiple inputs.
    
    Parameters:
        model (tf.keras.Model): The target model to be attacked.
        X_list (list of tf.Tensor): List of input data tensors.
        y_list (list of tf.Tensor): List of true label tensors.
        epsilon (float): Perturbation size.
        alpha (float): Step size for MIM.
        num_iter (int): Number of iterations for MIM.
        decay_factor (float): Decay factor for the step size.
        momentum (float): Momentum factor.
        batch_size (int): Batch size for processing inputs.
        
    Returns:
        adv_X_list (list of tf.Tensor): List of adversarial examples.
    """
    adv_X_list = [tf.identity(X) for X in X_list]  # Initialize adversarial examples with original inputs
    num_samples = X_list[0].shape[0]  # Assuming all inputs have the same number of samples
    
    for batch_start in range(0, num_samples, batch_size):
        batch_end = min(batch_start + batch_size, num_samples)
        batch_adv_X_list = [X[batch_start:batch_end] for X in adv_X_list]
        batch_y_list = [y[batch_start:batch_end] for y in y_list]
        
        perturbations = [tf.zeros_like(X) for X in batch_adv_X_list]  # Initialize perturbations with zeros
        
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                tape.watch(batch_adv_X_list)
                predictions = model(batch_adv_X_list)
                loss = sum([tf.keras.losses.binary_crossentropy(y, pred) for y, pred in zip(batch_y_list, predictions)])
            
            gradients = tape.gradient(loss, batch_adv_X_list)
            signed_grad = [tf.sign(grad) for grad in gradients]
            perturbations = [momentum * perturb + alpha * grad for perturb, grad in zip(perturbations, signed_grad)]
            batch_adv_X_list = [tf.clip_by_value(X + perturb, X - epsilon, X + epsilon) for X, perturb in zip(batch_adv_X_list, perturbations)]
            batch_adv_X_list = [tf.clip_by_value(X, 0, 1) for X in batch_adv_X_list]  # Clip to valid image range [0, 1]
            
            alpha *= decay_factor  # Decay step size
        
        # Update the adversarial examples back to the original list
        for i in range(len(adv_X_list)):
            indices = tf.range(batch_start, batch_end)
            adv_X_list[i] = tf.tensor_scatter_nd_update(adv_X_list[i], tf.expand_dims(indices, axis=1), batch_adv_X_list[i])
    
    return adv_X_list


In [ ]:
## MIM attack

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

#model = resnet18

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    '''
    [X_test,
              X_test_c], 
             [y_test,
              y_test_c]
              
              model.evaluate([X_test,X_test_c], [y_test,y_test_c])
              mim_attack(model, X_list, y_list, epsilon, alpha, 
              num_iter, decay_factor=1.0, momentum=0.9, batch_size=32)
    '''
    alpha = epsilon/4
    adv_X_train_list = mim_attack(model, [X_test,X_test_c], 
                          [y_test,y_test_c], epsilon, alpha, num_iter, decay_factor=1.0, momentum=0.9, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(y_test_c, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
model.save('best_mm_scan.keras')

In [ ]:
model.fit([X_train,
              images_train_br], 
             
             [labels_train_pneu,
              labels_train_br], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
             #verbose=0,
          validation_split=0.2)

In [ ]:
print(images_train_br.shape, labels_train_br.shape,
X_train.shape,X_test.shape, y_train.shape,y_test.shape)

In [ ]:
labels_train_br.shape, labels_train_pneu.shape

In [ ]:
from keras.utils import to_categorical

#labels_train_ct = to_categorical(labels_train_ct)
labels_train_br = to_categorical(labels_train_br)
labels_train_pneu = to_categorical(y_train)
labels_train_br.shape, labels_train_pneu.shape
#labels_test.shape, labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
print(images_train_br.shape, labels_train_br.shape,
X_train.shape,X_test.shape,labels_train_pneu.shape, y_train.shape,y_test.shape)

In [ ]:


import tensorflow as tf
from tensorflow.keras.applications import VGG16, MobileNet
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model

# Load pre-trained VGG16 model (excluding top layers)
input_shape = (128, 128, 3)
vgg16_base = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
# Apply combined attention noise to input data
'''
TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
'''
attention_noise_output = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(input_data)
print('attention_noise_output shape:', attention_noise_output.shape)

conv_layer_indices = [7, 12, 17]  # Indices of the convolutional layers in VGG16
x = attention_noise_output

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

for i, layer in enumerate(vgg16_base.layers):
    x = layer(x)
    #x = layer(x, name='inp')
    if i in conv_layer_indices:
        #x = SpatialAttentionLayer()(x)
        x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)

        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
         #                                            channel_noise_factor=1.0)(x)
        


print('x shape:', x.shape)
# Add the output of VGG16 with spatial attention noise and the output of combined attention noise

#combined_output = tf.concat([x, vgg16_output], axis=-1)
#print('combined_output shape:', combined_output.shape)
# Add additional layers for classification
flatten_layer = GlobalAveragePooling2D()(x)
dense_layer = Dense(128, activation='selu')(flatten_layer)
output_layer = Dense(2, activation='sigmoid')(dense_layer)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
## correct protect
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + spatial_noise
        channel_noise *= self.channel_noise_weight + channel_noise

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
## more correct protection
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='relu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:
## more correct protection
import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D, GlobalAveragePooling2D, Dense

class TrainableSpatialAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableSpatialAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.convolution = Conv2D(filters=1, kernel_size=(7, 7), activation='sigmoid', padding='same', 
                                  trainable=True)
        super(TrainableSpatialAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        attention_weights = self.convolution(inputs)
        return tf.multiply(inputs, attention_weights)

class TrainableChannelAttentionLayer(Layer):
    def __init__(self, **kwargs):
        super(TrainableChannelAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.global_avg_pooling = GlobalAveragePooling2D()
        self.dense1 = Dense(units=input_shape[-1] // 2, activation='selu', trainable=True)
        self.dense2 = Dense(units=input_shape[-1], activation='sigmoid', trainable=True)
        super(TrainableChannelAttentionLayer, self).build(input_shape)

    def call(self, inputs):
        avg_pool = self.global_avg_pooling(inputs)
        dense1_out = self.dense1(avg_pool)
        channel_attention_weights = self.dense2(dense1_out)
        channel_attention_weights = tf.expand_dims(tf.expand_dims(channel_attention_weights, 1), 1)
        return tf.multiply(inputs, channel_attention_weights)

class TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=1, **kwargs):
        super(TrainableCombinedAttentionLayer, self).__init__(**kwargs)
        self.spatial_attentions = [TrainableSpatialAttentionLayer() for _ in range(num_layers)]
        self.channel_attentions = [TrainableChannelAttentionLayer() for _ in range(num_layers)]
        
        # Trainable weights for noise factors
        self.spatial_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='spatial_noise_weight')
        self.channel_noise_weight = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='channel_noise_weight')
        
        self.spatial_noise_factor = spatial_noise_factor
        self.channel_noise_factor = channel_noise_factor

    def call(self, inputs):
        spatial_attention_output = inputs
        channel_attention_output = inputs

        for spatial_attention, channel_attention in zip(self.spatial_attentions, self.channel_attentions):
            # Spatial Attention
            spatial_attention_output = spatial_attention(spatial_attention_output)

            # Channel Attention
            channel_attention_output = channel_attention(channel_attention_output)

        # Generate spatial and channel noise
        spatial_noise = tf.random.normal(shape=tf.shape(spatial_attention_output), mean=0, 
                                         stddev=self.spatial_noise_factor)
        channel_noise = tf.random.normal(shape=tf.shape(channel_attention_output), mean=0, 
                                         stddev=self.channel_noise_factor)

        # Scale the noise with trainable weights
        spatial_noise *= self.spatial_noise_weight + (spatial_noise * self.spatial_noise_weight)
        channel_noise *= self.channel_noise_weight + (channel_noise * self.channel_noise_weight)

        # Add spatial attention noise
        spatial_attention_output += spatial_noise

        # Add channel attention noise
        channel_attention_output += channel_noise

        # Clip attention maps to ensure they are within the valid range [0, 1]
        spatial_attention_output = tf.clip_by_value(spatial_attention_output, 0, 1)
        channel_attention_output = tf.clip_by_value(channel_attention_output, 0, 1)

        # Combine attention mechanisms
        combined_attention1 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention2 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention3 = tf.multiply(spatial_attention_output, channel_attention_output)
        combined_attention4 = tf.multiply(combined_attention1, combined_attention2)
        combined_attention = tf.multiply(combined_attention3, combined_attention4)
        
        return tf.multiply(inputs, combined_attention)

# Example Usage:
'''model = tf.keras.models.Sequential([
    # Your CNN layers here
    # ...

    TrainableCombinedAttentionLayer(num_layers=3),
    # Add more layers if needed
    # ...
])'''


In [ ]:


import tensorflow as tf
from tensorflow.keras.applications import VGG16, MobileNet
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model

# Load pre-trained VGG16 model (excluding top layers)
input_shape = (128, 128, 3)
vgg16_base = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
# Apply combined attention noise to input data
'''
TrainableCombinedAttentionLayer(Layer):
    def __init__(self, spatial_noise_factor=1.0, channel_noise_factor=1.0, num_layers=3, **kwargs):
'''
attention_noise_output = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(input_data)
print('attention_noise_output shape:', attention_noise_output.shape)

conv_layer_indices = [7, 12, 17]  # Indices of the convolutional layers in VGG16
x = attention_noise_output

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

for i, layer in enumerate(vgg16_base.layers):
    x = layer(x)
    #x = layer(x, name='inp')
    if i in conv_layer_indices:
        #x = SpatialAttentionLayer()(x)
        x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)

        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
         #                                            channel_noise_factor=1.0)(x)
        


print('x shape:', x.shape)
# Add the output of VGG16 with spatial attention noise and the output of combined attention noise

#combined_output = tf.concat([x, vgg16_output], axis=-1)
#print('combined_output shape:', combined_output.shape)
# Add additional layers for classification
flatten_layer = GlobalAveragePooling2D()(x)
dense_layer = Dense(128, activation='selu')(flatten_layer)
output_layer = Dense(2, activation='sigmoid')(dense_layer)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
model.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,batch_size=128,
          validation_split=0.2)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
print(X_train_cervical.shape,X_test_cervical.shape,y_train_cervical.shape,y_test_cervical.shape,
X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape,
X_train_ham.shape,X_test_ham.shape,y_train_ham.shape,y_test_ham.shape,
X_train_aptos.shape,X_test_aptos.shape,y_train_aptos.shape,y_test_aptos.shape,
X_train_nick_brain.shape,X_test_nick_brain.shape,y_train_nick_brain.shape,y_test_nick_brain.shape,
X_train_nih_chest.shape,X_test_nih_chest.shape,y_train_nih_chest.shape,y_test_nih_chest.shape
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(inputs):
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs)

    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    return x
    
def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    
    inputs3 = Input(shape=input_shape)
    inputs4 = Input(shape=input_shape)
    
    inputs5 = Input(shape=input_shape)
    inputs6 = Input(shape=input_shape)
    inputs7 = Input(shape=input_shape)
    
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    x = tf.keras.layers.Concatenate(axis=-1)([inputs1, inputs2,inputs3,inputs4, inputs5, inputs6, inputs7])
    x1 = branch_ResNet18(x)
    
    '''x2 = branch_ResNet18(inputs2)
    
    x3 = branch_ResNet18(inputs3)
    x4 = branch_ResNet18(inputs4)
    
    x5 = branch_ResNet18(inputs5)
    x6 = branch_ResNet18(inputs6)
    
    x7 = branch_ResNet18(inputs7)
    
    x = tf.keras.layers.Concatenate(axis=-1)([x1, x2,
                                              x3, x4, x5,
                                              x6, x7
                                             ])'''
    # Global average pooling and fully connected layer
    
    out = GlobalAveragePooling2D()(x1)
    
    outputs1 = Dense(5, activation='softmax')(out)
    outputs2 = Dense(2, activation='sigmoid')(out)
    
    outputs3 = Dense(7, activation='softmax')(out)
    outputs4 = Dense(5, activation='softmax')(out)

    
    outputs5 = Dense(4, activation='softmax')(out)
    outputs6 = Dense(2, activation='sigmoid')(out)
    #outputs7 = Dense(2, activation='sigmoid')(out)


    # Create the model
    model = Model([inputs1, 
                   inputs2, 
                   inputs3, inputs4, inputs5, inputs6, #inputs7
                  ], [outputs1, outputs2, 
                                                                           outputs3, outputs4, outputs5, 
                                                                           outputs6, #outputs7
                                                                          ])
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet18.compile(optimizer='adam', loss=['categorical_crossentropy', 'binary_crossentropy',
                                        'categorical_crossentropy', 'categorical_crossentropy',
                                        'categorical_crossentropy', 'binary_crossentropy',
                                         #'binary_crossentropy'
                                        ], metrics=['accuracy', 'accuracy',
                                                        'accuracy','accuracy',
                                                        'accuracy', 'accuracy', #'accuracy'
                                                   ])

'''
X_train_cell_pneu.shape,X_test_cell_pneu.shape,y_train_cell_pneul.shape,y_test_cell_pneu.shape
y_train_cov2_1.shape,y_test_cov2_1.shape,y_train_nih_chest_1.shape,y_test_nih_chest_1.shape, y_train_cell_pneul_1.shape, y_test_cell_pneu_1.shape
'''
#images_train_br.shape,labels_train_br.shape,images_test_br.shape,labels_test_br.shape
#y_train_br.shape, y_test_br.shape, y_train_ct.shape, y_test_ct.shape
'''
X_train_cervical.shape,X_test_cervical.shape,y_train_cervical.shape,y_test_cervical.shape
X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape
X_train_ham.shape,X_test_ham.shape,y_train_ham.shape,y_test_ham.shape
X_train_aptos.shape,X_test_aptos.shape,y_train_aptos.shape,y_test_aptos.shape
X_train_nick_brain.shape,X_test_nick_brain.shape,y_train_nick_brain.shape,y_test_nick_brain.shape
X_train_nih_chest.shape,X_test_nih_chest.shape,y_train_nih_chest.shape,y_test_nih_chest.shape

y_train_cov2_1.shape,y_test_cov2_1.shape,y_train_nih_chest_1.shape,y_test_nih_chest_1.shape
'''

'''
X_train_nih_chest.shape,X_val_nih_chest.shape,y_train_nih_chest.shape,y_val_nih_chest.shape
X_train_nick_brain.shape,X_val_nick_brain.shape,y_train_nick_brain.shape,y_val_nick_brain.shape
X_train_aptos.shape,X_val_aptos.shape,y_train_aptos.shape,y_val_aptos.shape
X_train_cov2.shape,X_val_cov2.shape,y_train_cov2.shape,y_val_cov2.shape
X_train_cervical.shape,X_val_cervical.shape,y_train_cervical.shape,y_val_cervical.shape
X_train_cell_pneu.shape,X_val_cell_pneu.shape,y_train_cell_pneul.shape,y_val_cell_pneul.shape

'''
print('X_train_cervical:', X_train_cervical.shape)
print('X_train_cov2:', X_train_cov2.shape)
print('X_train_ham:', X_train_ham.shape)
print('X_train_aptos:', X_train_aptos.shape)
print('X_train_nick_brain:', X_train_nick_brain.shape)
print('X_train_nih_chest:', X_train_nih_chest.shape)
print('y_train_cervical:', y_train_cervical.shape)
print('y_train_cov2_1:', y_train_cov2_1.shape)
print('y_train_ham:', y_train_ham.shape)
print('y_train_aptos:', y_train_aptos.shape)

print('y_train_nick_brain:', y_train_nick_brain.shape)
print('y_train_nih_chest_1:', y_train_nih_chest_1.shape)

resnet18.fit([X_train_cervical, X_train_cov2, 
              X_train_ham, X_train_aptos, X_train_nick_brain, 
              X_train_nih_chest, #X_train_cell_pneu
             ], 
             [y_train_cervical, y_train_cov2_1, 
              y_train_ham, y_train_aptos, y_train_nick_brain, 
              y_train_nih_chest_1, #y_train_cell_pneul_1
             ], epochs=100, #batch_size=16, #callbacks = callbacks,
          validation_split=0.2)

# y_val_cov2_1.shape, y_val_nih_chest_1.shape, y_val_cell_pneul_1.shape

# Define your data generator function


In [ ]:
from keras.utils import to_categorical

y_train_cov2_1 = to_categorical(y_train_cov2)
y_test_cov2_1 = to_categorical(y_test_cov2)


In [ ]:

y_train_nih_chest_1 = to_categorical(y_train_nih_chest)
y_test_nih_chest_1 = to_categorical(y_test_nih_chest)

y_train_cov2_1.shape,y_test_cov2_1.shape,y_train_nih_chest_1.shape,y_test_nih_chest_1.shape

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(inputs):
    

    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    return x
    
def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    
    inputs3 = Input(shape=input_shape)
    inputs4 = Input(shape=input_shape)
    
    inputs1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs1)
    
    inputs2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs2)
    
    inputs3 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs3)
    
    inputs4 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs4)
    
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    x = tf.keras.layers.Concatenate(axis=-1)([inputs1, inputs2, inputs3, inputs4])
    x = branch_ResNet18(x)
    
    #x2 = branch_ResNet18(inputs2)
    
    #x3 = branch_ResNet18(inputs3)
    #x4 = branch_ResNet18(inputs4)
    
    
    '''
    x = tf.keras.layers.Concatenate(axis=-1)([x1, x2,
                                              x3, x4
                                             ])
                                             '''
    # Global average pooling and fully connected layer
    
    out = GlobalAveragePooling2D()(x)
    #out2 = tf.keras.layers.GlobalMaxPooling2D()(x)
    
    #x = tf.keras.layers.Add()([out1, out2])
    
    ##x = Dense(128, activation='selu')(x)
    
    outputs1 = Dense(5, activation='softmax')(out)
    outputs2 = Dense(7, activation='softmax')(out)
    
    outputs3 = Dense(5, activation='softmax')(out)
    outputs4 = Dense(2, activation='sigmoid')(out)
    


    # Create the model
    model = Model([inputs1, inputs2, inputs3, inputs4], 
                  [outputs1, outputs2, outputs3, outputs4])
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet18.compile(optimizer='adam', loss=['categorical_crossentropy','categorical_crossentropy', 
                                         'categorical_crossentropy','binary_crossentropy'
                                        ], metrics=['accuracy','accuracy', 
                                                    'accuracy','accuracy'
                                                   ])

#images_train_br.shape,labels_train_br.shape,images_test_br.shape,labels_test_br.shape
#y_train_br.shape, y_test_br.shape, y_train_ct.shape, y_test_ct.shape
'''
X_train_cervical.shape,X_test_cervical.shape,y_train_cervical.shape,y_test_cervical.shape
X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape
X_train_ham.shape,X_test_ham.shape,y_train_ham.shape,y_test_ham.shape
X_train_aptos.shape,X_test_aptos.shape,y_train_aptos.shape,y_test_aptos.shape
X_train_nick_brain.shape,X_test_nick_brain.shape,y_train_nick_brain.shape,y_test_nick_brain.shape
X_train_nih_chest.shape,X_test_nih_chest.shape,y_train_nih_chest.shape,y_test_nih_chest.shape

y_train_cov2_1.shape,y_test_cov2_1.shape,y_train_nih_chest_1.shape,y_test_nih_chest_1.shape
print(X_train_cell_pneu.shape,X_test_cell_pneu.shape,y_train_cell_pneul.shape,y_test_cell_pneu.shape,

y_train_cov2_1.shape,y_test_cov2_1.shape,y_train_nih_chest_1.shape,y_test_nih_chest_1.shape, 
y_train_cell_pneul_1.shape, y_test_cell_pneu_1.shape)
'''

resnet18.fit([X_train_cervical, X_train_ham, X_train_aptos, X_train_cell_pneu], 
             
             [y_train_cervical, y_train_ham, y_train_aptos, y_train_cell_pneul_1], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(x):
    
    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    return x
    
def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    
    #inputs3 = Input(shape=input_shape)
    
    inputs = tf.keras.layers.Concatenate(axis=-1)([inputs1, inputs2
                                                   #, inputs3
                                                  ])
    
    inputs = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs)
    
    
    x1 = branch_ResNet18(inputs)
    x1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x1)
    
    x2 = branch_ResNet18(inputs)
    x2 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x2)
    
    #x3 = branch_ResNet18(inputs)
    
    x = tf.keras.layers.Concatenate(axis=-1)([x1, x2])
    
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters=512, kernel_size=(3, 3), padding='same', activation = 'relu')(x)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0,num_layers=1)(x)
    
    
    out = tf.keras.layers.GlobalAveragePooling2D()(x)
    
    
    outputs1 = Dense(2, activation='sigmoid')(out)
    
    outputs2 = Dense(2, activation='sigmoid')(out)
    #outputs3 = Dense(5, activation='softmax')(out)
    
    #outputs5 = Dense(7, activation='softmax')(out)
    


    # Create the model
    model = Model([inputs1, inputs2], 
                  [outputs1, outputs2])
    return model

# Instantiate the ResNet-18 model
resnet18 = build_resnet18()

# Display the model summary
#resnet18.summary()


resnet18.compile(optimizer='adam', loss=['binary_crossentropy', 
                                         'binary_crossentropy',#'categorical_crossentropy'
                                        ], metrics=['accuracy','accuracy', 
                                                    #'accuracy'
                                                   ])

#X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape
'''
print(images_train_br.shape, labels_train_br.shape)
print(images_train_ct.shape, labels_train_ct.shape)
'''

resnet18.fit([X_train_cell_pneu,
              images_train_br], 
             
             [y_train_cell_pneul_1,
              labels_train_br], 
             epochs=100, 
             #batch_size=16, #callbacks = callbacks,
             #verbose=0,
          validation_split=0.2)

In [ ]:
labels_train_br.shape, labels_train_ct.shape

In [ ]:
#X_train_cov2.shape,X_test_cov2.shape,y_train_cov2.shape,y_test_cov2.shape

In [ ]:
resnet18.fit([X_train_cell_pneu, #X_train_cervical, 
              images_train_br], 
             
             [y_train_cell_pneul_1, #y_train_cervical, 
              labels_train_br], 
             epochs=100, ##verbose=0,
             #batch_size=16, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
from keras.utils import to_categorical

#labels_train_ct = to_categorical(labels_train_ct)
labels_train_br = to_categorical(labels_train_br)

labels_train_br.shape, labels_train_ct.shape
#labels_test.shape, labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
labels_train_br.shape, labels_train_ct.shape

In [ ]:
labels_test.shape, labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
from keras.utils import to_categorical

#labels_test = to_categorical(labels_test_kv)
labels_test_br = to_categorical(labels_test_br)
#y_test_cell_pneu = to_categorical(y_test_cell_pneu)
#labels_train_br.shape, labels_train_ct.shape
#labels_test.shape, 
labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
y_test_cell_pneu = to_categorical(y_test_cell_pneu)
y_test_cell_pneu.shape

In [ ]:
labels_test = to_categorical(labels_test)
labels_test.shape, labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
#labels_test.shape, 
labels_test_br.shape, y_test_cell_pneu.shape

In [ ]:
'''X_train_cell_pneu.shape,X_test_cell_pneu.shape,y_train_cell_pneul.shape,y_test_cell_pneu.shape
images_test.shape, labels_test.shape
images_test_br.shape, labels_test_br.shape
'''
'''
resnet18.fit([X_train_cell_pneu, #X_train_cervical, 
              images_train_br, images_train_ct], 

'''
resnet18.evaluate([X_test_cell_pneu,images_test_br], [y_test_cell_pneu, labels_test_br],
                 #batch_size=8
                 )

In [ ]:
labels_test_br.shape

In [ ]:
'''gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Currently, memory growth needs to be the same across GPUs
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(e)
'''

In [ ]:
import tensorflow as tf

def pgd_attack(model, X_list, y_list, epsilon, alpha, num_iter, batch_size=32):
    """
    PGD adversarial attack on the model with multiple inputs.
    
    Parameters:
        model (tf.keras.Model): The target model to be attacked.
        X_list (list of tf.Tensor): List of input data tensors.
        y_list (list of tf.Tensor): List of true label tensors.
        epsilon (float): Perturbation size.
        alpha (float): Step size for PGD.
        num_iter (int): Number of iterations for PGD.
        batch_size (int): Batch size for processing inputs.
        
    Returns:
        adv_X_list (list of tf.Tensor): List of adversarial examples.
    """
    adv_X_list = [tf.identity(X) for X in X_list]  # Initialize adversarial examples with original inputs
    num_samples = X_list[0].shape[0]  # Assuming all inputs have the same number of samples
    
    for batch_start in range(0, num_samples, batch_size):
        batch_end = min(batch_start + batch_size, num_samples)
        batch_adv_X_list = [X[batch_start:batch_end] for X in adv_X_list]
        batch_y_list = [y[batch_start:batch_end] for y in y_list]
        
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                tape.watch(batch_adv_X_list)
                predictions = model(batch_adv_X_list)
                loss = sum([tf.keras.losses.binary_crossentropy(y, pred) for y, pred in zip(batch_y_list, predictions)])
            
            gradients = tape.gradient(loss, batch_adv_X_list)
            signed_grad = [tf.sign(grad) for grad in gradients]
            perturbations = [alpha * grad for grad in signed_grad]
            batch_adv_X_list = [tf.clip_by_value(X + perturbation, X - epsilon, X + epsilon) for X, perturbation in zip(batch_adv_X_list, perturbations)]
            batch_adv_X_list = [tf.clip_by_value(X, 0, 1) for X in batch_adv_X_list]  # Clip to valid image range [0, 1]
        
        # Update the adversarial examples back to the original list
        for i in range(len(adv_X_list)):
            indices = tf.range(batch_start, batch_end)
            adv_X_list[i] = tf.tensor_scatter_nd_update(adv_X_list[i], tf.expand_dims(indices, axis=1), batch_adv_X_list[i])
    
    return adv_X_list

# Assuming you have a TensorFlow model defined and compiled
# model = ...

# Assuming X_train, X_train_c, y_train, y_train_c are TensorFlow tensors
# Define your parameters
epsilon = 1.0
alpha = 0.25
num_iter = 20
batch_size = 10
model = resnet18
# Generate adversarial examples
adv_X_train_list = pgd_attack(model, [images_test_br, images_test_ct], 
                              [y_test_br, y_test_ct], epsilon, alpha, num_iter, batch_size)


In [ ]:
import tensorflow as tf

def pgd_attack(model, X_list, y_list, epsilon, alpha, num_iter, batch_size=32):
    """
    PGD adversarial attack on the model with multiple inputs.
    
    Parameters:
        model (tf.keras.Model): The target model to be attacked.
        X_list (list of tf.Tensor): List of input data tensors.
        y_list (list of tf.Tensor): List of true label tensors.
        epsilon (float): Perturbation size.
        alpha (float): Step size for PGD.
        num_iter (int): Number of iterations for PGD.
        batch_size (int): Batch size for processing inputs.
        
    Returns:
        adv_X_list (list of tf.Tensor): List of adversarial examples.
    """
    adv_X_list = [tf.identity(X) for X in X_list]  # Initialize adversarial examples with original inputs
    num_samples = X_list[0].shape[0]  # Assuming all inputs have the same number of samples
    
    for batch_start in range(0, num_samples, batch_size):
        batch_end = min(batch_start + batch_size, num_samples)
        batch_adv_X_list = [X[batch_start:batch_end] for X in adv_X_list]
        batch_y_list = [y[batch_start:batch_end] for y in y_list]
        
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                tape.watch(batch_adv_X_list)
                predictions = model(batch_adv_X_list)
                loss = sum([tf.keras.losses.binary_crossentropy(y, pred) for y, pred in zip(batch_y_list, predictions)])
            
            gradients = tape.gradient(loss, batch_adv_X_list)
            signed_grad = [tf.sign(grad) for grad in gradients]
            perturbations = [alpha * grad for grad in signed_grad]
            batch_adv_X_list = [tf.clip_by_value(X + perturbation, X - epsilon, X + epsilon) for X, perturbation in zip(batch_adv_X_list, perturbations)]
            batch_adv_X_list = [tf.clip_by_value(X, 0, 1) for X in batch_adv_X_list]  # Clip to valid image range [0, 1]
        
        # Update the adversarial examples back to the original list
        for i in range(len(adv_X_list)):
            indices = tf.range(batch_start, batch_end)
            adv_X_list[i] = tf.tensor_scatter_nd_update(adv_X_list[i], tf.expand_dims(indices, axis=1), batch_adv_X_list[i])
    
    return adv_X_list


In [ ]:

import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

model = resnet18

# Define epsilon values
#epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test_cell_pneu, images_test_br], 
                          [y_test_cell_pneu, labels_test_br], epsilon, alpha, num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test_cell_pneu, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(labels_test_br, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    '''y_pred_2 = np.argmax(y_pred[2], axis=1)
    y_test_2 = np.argmax(labels_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_2, y_test_2)
    precision_br = precision_score(y_pred_2, y_test_2)
    recall_br = recall_score(y_pred_2, y_test_2)
    f1_br = f1_score(y_pred_2, y_test_2)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for COVID-19 classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    '''
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

model = resnet18

# Define epsilon values
epsilon_values = [0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test_cell_pneu, images_test_br], 
                          [y_test_cell_pneu, labels_test_br], epsilon, alpha, num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test_cell_pneu, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(labels_test_br, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    '''y_pred_2 = np.argmax(y_pred[2], axis=1)
    y_test_2 = np.argmax(labels_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_2, y_test_2)
    precision_br = precision_score(y_pred_2, y_test_2)
    recall_br = recall_score(y_pred_2, y_test_2)
    f1_br = f1_score(y_pred_2, y_test_2)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for COVID-19 classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    '''
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

model = resnet18

# Define epsilon values
epsilon_values = [0.008, 0.016, 0.0314, 0.0627, 0.1, 0.125]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 100

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test_cell_pneu, images_test_br], 
                          [y_test_cell_pneu, labels_test_br], epsilon, alpha, num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test_cell_pneu, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(labels_test_br, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    '''y_pred_2 = np.argmax(y_pred[2], axis=1)
    y_test_2 = np.argmax(labels_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_2, y_test_2)
    precision_br = precision_score(y_pred_2, y_test_2)
    recall_br = recall_score(y_pred_2, y_test_2)
    f1_br = f1_score(y_pred_2, y_test_2)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for COVID-19 classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    '''
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

model = resnet18

# Define epsilon values
#epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test_cell_pneu, images_test_br, images_test], 
                          [y_test_cell_pneu, labels_test_br, labels_test], epsilon, alpha, num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1], adv_X_train_list[2]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test_cell_pneu, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(labels_test_br, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    y_pred_2 = np.argmax(y_pred[2], axis=1)
    y_test_2 = np.argmax(labels_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_2, y_test_2)
    precision_br = precision_score(y_pred_2, y_test_2)
    recall_br = recall_score(y_pred_2, y_test_2)
    f1_br = f1_score(y_pred_2, y_test_2)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for COVID-19 classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size

model = resnet18

# Define epsilon values
epsilon_values = [0, 0.01,0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,0.2,0.3]

#epsilon_values = [0.1]
batch_size = 10

num_iter = 10

#resnet18.evaluate([X_test_cell_pneu,images_test_br, images_test], [y_test_cell_pneu, labels_test_br, labels_test])
for epsilon in epsilon_values:
    #adversarial_examples = []
    alpha = epsilon/4
    adv_X_train_list = pgd_attack(model, [X_test_cell_pneu, images_test_br, images_test], 
                          [y_test_cell_pneu, labels_test_br, labels_test], epsilon, alpha, num_iter, batch_size)

    #resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])
    
    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict([adv_X_train_list[0], adv_X_train_list[1], adv_X_train_list[2]]))
    #_pred_binary = y_pred >= 0.5
    #_pred_binary = np.array(y_pred_binary, dtype='int32')
    
    y_pred_0 = np.argmax(y_pred[0], axis=1)
    y_test_0 = np.argmax(y_test_cell_pneu, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_0, y_test_0)
    precision_br = precision_score(y_pred_0, y_test_0)
    recall_br = recall_score(y_pred_0, y_test_0)
    f1_br = f1_score(y_pred_0, y_test_0)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for Cell Pneumonia Chest X-ray classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    #print("AUC Score:", auc_br)
    
    
    y_pred_1 = np.argmax(y_pred[1], axis=1)
    y_test_1 = np.argmax(labels_test_br, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_ct = accuracy_score(y_pred_1, y_test_1)
    precision_ct = precision_score(y_pred_1, y_test_1)
    recall_ct = recall_score(y_pred_1, y_test_1)
    f1_ct = f1_score(y_pred_1, y_test_1)
    #auc_ct = roc_auc_score(y_pred_1, y_test_ct_1) * 100

    # Store results for the current epsilon
    print('for Brain tumours classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_ct)
    print("Precision:",precision_ct)
    print("Recall:", recall_ct)
    print("F1 Score:", f1_ct)
    #print("AUC Score:", auc_ct)
    
    y_pred_2 = np.argmax(y_pred[2], axis=1)
    y_test_2 = np.argmax(labels_test, axis=1)
    # Calculate evaluation metrics for the current epsilon
    
    accuracy_br = accuracy_score(y_pred_2, y_test_2)
    precision_br = precision_score(y_pred_2, y_test_2)
    recall_br = recall_score(y_pred_2, y_test_2)
    f1_br = f1_score(y_pred_2, y_test_2)
    #auc_br = roc_auc_score(y_pred_0, y_test_br_0) * 100

    # Store results for the current epsilon
    print('for COVID-19 classifiaction:')
    print("Epsilon:", epsilon)
    print("Accuracy:",accuracy_br)
    print("Precision:",precision_br)
    print("Recall:", recall_br)
    print("F1 Score:", f1_br)
    
    
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
adv_X_train_list[0].shape

In [ ]:
resnet18.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model

def residual_block(x, filters, strides=(1, 1), use_projection=False):
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    
    x = Activation('relu')(x)
    return x

def branch_ResNet18(inputs):
    '''x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs)
    '''
    x = inputs
    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    return x
    
def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs1 = Input(shape=input_shape)
    inputs2 = Input(shape=input_shape)
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    
    x1 = branch_ResNet18(inputs1)
    x2 = branch_ResNet18(inputs2)
    
    x = tf.keras.layers.Concatenate(axis=-1)([x1, x2])
    # Global average pooling and fully connected layer
    x = GlobalAveragePooling2D()(x)
    ##x = Dense(128, activation='selu')(x)
    outputs1 = Dense(2, activation='sigmoid')(x)
    outputs2 = Dense(2, activation='sigmoid')(x)

    # Create the model
    model = Model([inputs1, inputs2], [outputs1, outputs2])
    return model

# Instantiate the ResNet-18 model
resnet181 = build_resnet18()

# Display the model summary
#resnet18.summary()
resnet181.compile(optimizer='adam', loss=['binary_crossentropy', 'binary_crossentropy'], metrics=['accuracy',
                                                                                                'accuracy'])

#images_train_br.shape,labels_train_br.shape,images_test_br.shape,labels_test_br.shape
#y_train_br.shape, y_test_br.shape, y_train_ct.shape, y_test_ct.shape

resnet181.fit([images_train_br, images_train_ct], [y_train_br, y_train_ct], epochs=10, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
resnet181.evaluate([images_test_br, images_test_ct], [y_test_br, y_test_ct])

In [ ]:
import tensorflow as tf

def pgd_attack(model, X_list, y_list, epsilon, alpha, num_iter, batch_size=32):
    """
    PGD adversarial attack on the model with multiple inputs.
    
    Parameters:
        model (tf.keras.Model): The target model to be attacked.
        X_list (list of tf.Tensor): List of input data tensors.
        y_list (list of tf.Tensor): List of true label tensors.
        epsilon (float): Perturbation size.
        alpha (float): Step size for PGD.
        num_iter (int): Number of iterations for PGD.
        batch_size (int): Batch size for processing inputs.
        
    Returns:
        adv_X_list (list of tf.Tensor): List of adversarial examples.
    """
    adv_X_list = [tf.identity(X) for X in X_list]  # Initialize adversarial examples with original inputs
    num_samples = X_list[0].shape[0]  # Assuming all inputs have the same number of samples
    
    for batch_start in range(0, num_samples, batch_size):
        batch_end = min(batch_start + batch_size, num_samples)
        batch_adv_X_list = [X[batch_start:batch_end] for X in adv_X_list]
        batch_y_list = [y[batch_start:batch_end] for y in y_list]
        
        for _ in range(num_iter):
            with tf.GradientTape() as tape:
                tape.watch(batch_adv_X_list)
                predictions = model(batch_adv_X_list)
                loss = sum([tf.keras.losses.binary_crossentropy(y, pred) for y, pred in zip(batch_y_list, predictions)])
            
            gradients = tape.gradient(loss, batch_adv_X_list)
            signed_grad = [tf.sign(grad) for grad in gradients]
            perturbations = [alpha * grad for grad in signed_grad]
            batch_adv_X_list = [tf.clip_by_value(X + perturbation, X - epsilon, X + epsilon) for X, perturbation in zip(batch_adv_X_list, perturbations)]
            batch_adv_X_list = [tf.clip_by_value(X, 0, 1) for X in batch_adv_X_list]  # Clip to valid image range [0, 1]
        
        # Update the adversarial examples back to the original list
        for i in range(len(adv_X_list)):
            indices = tf.range(batch_start, batch_end)
            adv_X_list[i] = tf.tensor_scatter_nd_update(adv_X_list[i], tf.expand_dims(indices, axis=1), batch_adv_X_list[i])
    
    return adv_X_list

# Assuming you have a TensorFlow model defined and compiled
# model = ...

# Assuming X_train, X_train_c, y_train, y_train_c are TensorFlow tensors
# Define your parameters
epsilon = 1.0
alpha = 0.25
num_iter = 20
batch_size = 10
model = resnet181
# Generate adversarial examples
adv_X_train_list = pgd_attack(model, [images_test_br, images_test_ct], 
                              [y_test_br, y_test_ct], epsilon, alpha, num_iter, batch_size)


In [ ]:
resnet181.evaluate([adv_X_train_list[0], adv_X_train_list[1]], [y_test_br, y_test_ct])

In [ ]:
resnet18.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,
          validation_split=0.2)

In [ ]:
'''model1 = model
model = resnet18'''

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn = resnet18,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(resnet18.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import tensorflow as tf

# Load the model
loaded_model = tf.keras.models.load_model("your_model_name.tf")

# Now you can use the loaded model for inference or further training

In [ ]:
import tensorflow as tf
loaded_model.save("resnet_train_sca_covid_ct1.h5")

In [ ]:
loaded_model1 = tf.keras.models.load_model("resnet_train_sca_covid_ct1.h5", 
                                          custom_objects = {'TrainableCombinedAttentionLayer': TrainableCombinedAttentionLayer,
                                                            'CombinedAttentionNoiseLayer': CombinedAttentionNoiseLayer})

In [ ]:
loaded_model1.evaluate(images_test, y_test_one_hot)

In [ ]:
loaded_model.evaluate(images_test, y_test_one_hot)

In [ ]:
def residual_block(x, filters, strides=(1, 1), use_projection=False):
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    shortcut = x

    # Define the first convolutional layer of the block
    x = Conv2D(filters=filters, kernel_size=(3, 3), strides=strides, padding='same', 
               activation = 'relu')(x)
    x = BatchNormalization()(x)
    #x = Activation('relu')(x)

    # Define the second convolutional layer of the block
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = Conv2D(filters=filters, kernel_size=(3, 3), padding='same')(x)
    x = BatchNormalization()(x)

    # If the stride is not (1, 1), the dimensions need to be adjusted
    if strides != (1, 1) or use_projection:
        
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(shortcut)
        shortcut = BatchNormalization()(shortcut)

    # Add the shortcut (identity connection)
    
    x = tf.keras.layers.add([x, shortcut])
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    x = Activation('relu')(x)
    return x

def build_resnet18(input_shape=(128, 128, 3), num_classes=2):
    inputs = Input(shape=input_shape)
    #input_data = Input(shape=input_shape, name='input_data')
    # Initial convolutional layer
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                   #                     num_layers=1
                                   )(inputs)

    x = Conv2D(filters=64, kernel_size=(7, 7), strides=(2, 2), padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = MaxPooling2D(pool_size=(3, 3), strides=(2, 2), padding='same')(x)

    # Stack of residual blocks
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=64)
    
    x = residual_block(x, filters=128, strides=(2, 2), use_projection=True)

    x = residual_block(x, filters=128)
    
    x = residual_block(x, filters=256, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=256)
    
    x = residual_block(x, filters=512, strides=(2, 2), use_projection=True)
    
    x = residual_block(x, filters=512)
    
    x = TrainableCombinedAttentionLayer(spatial_noise_factor=1.0, 
                                                     channel_noise_factor=1.0,
                                                        num_layers=1)(x)
    
    # Global average pooling and fully connected layer
    x = GlobalAveragePooling2D()(x)
    ##x = Dense(128, activation='selu')(x)
    x = Dropout(0.1)(x)
    outputs = Dense(2, activation='sigmoid')(x)

    # Create the model
    model = Model(inputs, outputs)
    return model

# Instantiate the ResNet-18 model



In [ ]:
import numpy as np
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import KLDivergence

# Set a random seed for reproducibility
np.random.seed(42)

def create_ensemble(num_models, input_shape=(128, 128, 3), num_classes=2, dropout_rate=0.1):
    ensemble_models = []
    
    for _ in range(num_models):
        #resnet18 = build_resnet18()
        model = build_resnet18()  # Assuming ResNet18 is defined elsewhere
        ensemble_models.append(model)
    
    return ensemble_models

# Function to perform Monte Carlo Dropout inference

# Example usage
input_shape = (128, 128, 3)
num_classes = 2
num_models = 5
dropout_rate = 0.1

ensemble_models = create_ensemble(num_models, input_shape, num_classes, dropout_rate)

# Train each model in the ensemble
for i, model in enumerate(ensemble_models):
    print("Training Model", i)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Define checkpoint callback for each model
    checkpoint = ModelCheckpoint(f"best_model_el_sca_covid_ct1_{i}.tf", monitor='val_loss', 
                                 verbose=1, save_best_only=True, mode='min')
    
    #model.fit(X_train1, y_train1, epochs=200, callbacks=[checkpoint], 
              #validation_split=0.2, verbose=0)
    model.fit(images_train, y_train_one_hot,
                 epochs=100,
                 validation_split=0.2,callbacks = [checkpoint], verbose=0,
                 shuffle= False)
    model.fit(images_train, y_train_one_hot,
                 epochs=100,
                 validation_split=0.2,callbacks = [checkpoint], verbose=0,
                 shuffle= False)
    model.fit(images_train, y_train_one_hot,
                 epochs=100,
                 validation_split=0.2,callbacks = [checkpoint], verbose=0,
                 shuffle= False)

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import KLDivergence

e0 = load_model('/working/best_model_el_sca_covid_ct1_0.tf')
e1 = load_model('/working/best_model_el_sca_covid_ct1_1.tf')
e2 = load_model('/working/best_model_el_sca_covid_ct1_2.tf')
e0.evaluate(images_test, y_test_one_hot)
e1.evaluate(images_test, y_test_one_hot)
e2.evaluate(images_test, y_test_one_hot)


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import KLDivergence

e0 = load_model('/working/best_model_el_sca_covid_ct1_0.tf')
e1 = load_model('/working/best_model_el_sca_covid_ct1_1.tf')
e2 = load_model('/working/best_model_el_sca_covid_ct1_2.tf')
e3 = load_module('/working/best_model_el_sca_covid_ct1_1.tf')
e4 = load_module('/working/best_model_el_sca_covid_ct1_1.tf')
e0.evaluate(images_test, y_test_one_hot)
e1.evaluate(images_test, y_test_one_hot)
e2.evaluate(images_test, y_test_one_hot)
e3.evaluate(images_test, y_test_one_hot)
e4.evaluate(images_test, y_test_one_hot)


In [ ]:
e0.save('best_model_el_sca_covid_ct1_0.h5')
e1.save('best_model_el_sca_covid_ct1_1.h5')
e2.save('best_model_el_sca_covid_ct1_2.h5')

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.losses import KLDivergence
ensemble_models_0 = load_model('/working/best_model_0.h5', custom_objects = {'DeeperAttentionLayer': DeeperAttentionLayer})
ensemble_models_1 = load_model('/working/best_model_1.h5', custom_objects = {'DeeperAttentionLayer': DeeperAttentionLayer})
ensemble_models_2 = load_model('/working/best_model_2.h5', custom_objects = {'DeeperAttentionLayer': DeeperAttentionLayer})
ensemble_models_3 = load_model('/working/best_model_3.h5', custom_objects = {'DeeperAttentionLayer': DeeperAttentionLayer})
ensemble_models_4 = load_model('/working/best_model_4.h5', custom_objects = {'DeeperAttentionLayer': DeeperAttentionLayer})
'''ensemble_models_5 = load_model('/input/vgg16-data-deep-el-model/vgg16_ensemble_models5.h5', custom_objects={'distillation_loss': distillation_loss})
ensemble_models_6 = load_model('/input/vgg16-data-deep-el-model/vgg16_ensemble_models6.h5', custom_objects={'distillation_loss': distillation_loss})
ensemble_models_7 = load_model('/input/updated-vgg16-el-models/ensemble_models7.h5', custom_objects={'distillation_loss': distillation_loss})
ensemble_models_8 = load_model('/input/updated-vgg16-el-models/ensemble_models8.h5', custom_objects={'distillation_loss': distillation_loss})
ensemble_models_9 = load_model('/input/updated-vgg16-el-models/ensemble_models9.h5', custom_objects={'distillation_loss': distillation_loss})
'''
ensemble_models_0.evaluate(ts_gen)
ensemble_models_1.evaluate(ts_gen)
ensemble_models_2.evaluate(ts_gen)
ensemble_models_3.evaluate(ts_gen)
ensemble_models_4.evaluate(ts_gen)
'''ensemble_models_5.evaluate([X_test], y_test)
ensemble_models_6.evaluate([X_test], y_test)
ensemble_models_7.evaluate([X_test], y_test)
ensemble_models_8.evaluate([X_test], y_test)
ensemble_models_9.evaluate([X_test], y_test)'''

In [ ]:
model.fit(images_train, y_train_one_hot, epochs=100, #callbacks = callbacks,batch_size=128,
          validation_split=0.2)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Layer, Conv2D, DepthwiseConv2D

# Assume you have defined CombinedAttentionNoiseLayer class

# Load pre-trained MobileNet model (excluding top layers)
input_shape = (128, 128, 3)
mobilenet_base = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of MobileNet
for layer in mobilenet_base.layers:
    layer.trainable = False

# Specify the indices of layers where you want to add combined attention noise
#
# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
#attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, 
 #                                                    channel_noise_factor=0.1)(input_data)

# Apply combined attention noise before specified layers
x = input_data
for i, layer in enumerate(mobilenet_base.layers):
    #if i in attention_indices:
        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, channel_noise_factor=1.0)(x)
    x = layer(x)

# Add additional layers for classification
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='selu')(x)
output_layer = Dense(2, activation='sigmoid')(x)

# Create the final model
model1 = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()
#model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model1.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)

# Continue with training the model...


In [ ]:
y_train_one_hot.shape

In [ ]:
input_shape = (128, 128, 3)
vgg16_base = MobileNet(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
# Apply combined attention noise to input data
attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, 
                                                     channel_noise_factor=0.1)(input_data)
print('attention_noise_output shape:', attention_noise_output.shape)

#vgg16_output = vgg16_base(attention_noise_output)
#print('vgg16_output shape:', vgg16_output.shape)
# Apply spatial attention noise to specific convolutional layers (e.g., the 6th, 12th, and 18th convolutional layers)

conv_layer_indices = [2, 7, 12, 17]  # Indices of the convolutional layers in VGG16
x = input_data

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

'''class SpatialAttentionLayer1(tf.keras.layers.Layer):
    def __init__(self, spatial_noise_factor=1.0, **kwargs):
        super(SpatialAttentionLayer1, self).__init__(**kwargs)
        self.spatial_noise_factor = spatial_noise_factor
        self.convolution = Conv2D(filters=1, kernel_size=(1, 1), activation='sigmoid', padding='same')

    def build(self, input_shape):
        super(SpatialAttentionLayer1, self).build(input_shape)

    def call(self, inputs, training=None):
        attention_weights = self.convolution(inputs)
        
        if training:
            # Add spatial attention noise during training
            attention_weights += tf.random.normal(shape=tf.shape(attention_weights),
                                                 mean=0, stddev=self.spatial_noise_factor)
        
        return tf.multiply(inputs, attention_weights)
'''
for i, layer in enumerate(vgg16_base.layers):
    x = layer(x)
    #x = layer(x, name='inp')
    #if i in conv_layer_indices:
        #x = SpatialAttentionLayer()(x)
        #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
         #                                            channel_noise_factor=1.0)(x)
        


print('x shape:', x.shape)
# Add the output of VGG16 with spatial attention noise and the output of combined attention noise

#combined_output = tf.concat([x, vgg16_output], axis=-1)
#print('combined_output shape:', combined_output.shape)
# Add additional layers for classification
flatten_layer = GlobalAveragePooling2D()(x)
dense_layer = Dense(128, activation='relu')(flatten_layer)
output_layer = Dense(2, activation='sigmoid')(dense_layer)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary
model.summary()

# Continue with training the model...
# Continue with training the model...


In [ ]:

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)

In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=0.01,
        eps_iter=0.001,
        nb_iter=10,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Load the pre-trained VGG16 model with ImageNet weights (include_top=False for feature extraction)
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the layers of the pre-trained model
for layer in base_model.layers:
    layer.trainable = False

# Add custom layers for binary classification
model = models.Sequential()
model.add(base_model)
model.add(layers.Flatten())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(2, activation='sigmoid'))  # Binary classification, so use 'sigmoid' activation

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
lr = 0.005

# Choose an optimizer and pass the learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Train the model
#model.fit(images_train, y_train_one_hot, epochs=10, batch_size=32, validation_split=0.2)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)
#model.summary()

# Display the model summary
#model.summary()


In [ ]:
# Assuming you have training and validation datasets (X_train, y_train, X_val, y_val)


# Evaluate the model
test_loss, test_acc = model.evaluate(images_test, y_test_one_hot)
print(f'Test Accuracy: {test_acc * 100:.2f}%')


In [ ]:
model1 = loaded_model = tf.keras.models.load_model('/working/best_models1.h5')
test_loss, test_acc = model1.evaluate(images_test, y_test_one_hot)
print(f'Test Accuracy: {test_acc * 100:.2f}%')

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, GlobalAveragePooling2D, ReLU, Add, AveragePooling2D, Flatten, Dense

import tensorflow as tf
from tensorflow.keras.layers import Input, MaxPooling2D, Conv2D, BatchNormalization, ReLU, Add, AveragePooling2D, Flatten, Dense

def residual_block(x, filters, strides=(1, 1), use_attention=True):
    filters1, filters2, filters3 = filters
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, channel_noise_factor=0.1)(x)
    
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters3, (1, 1), strides=strides, padding='same'
               , use_bias=False, kernel_initializer='he_normal')(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
     #                               channel_noise_factor=0.01)(x)
    
    #x = BatchNormalization()(x)
    #x = BatchNormalization()(x)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
   
    x_shortcut = x
    #print('x_shortcut shape:', x_shortcut.shape)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    # First block
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters1, (3, 3), strides=strides, padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    x = BatchNormalization()(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
           #                         channel_noise_factor=0.3)(x)
   
    # Second block
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters2, (3, 3), padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
     #                               channel_noise_factor=0.01)(x)
    
    x = BatchNormalization()(x)
    
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    
    # Third block
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters3, (3, 3), padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
     #                               channel_noise_factor=0.01)(x)
    
    x = BatchNormalization()(x)
    
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, channel_noise_factor=0.1)(x)
    
    # Adjust shortcut connection dimensions to match main path
    if strides != (1, 1) or x.shape[-1] != filters3:
        #x_shortcut = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
         #                           channel_noise_factor=1.0)(x_shortcut)
        x_shortcut = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x_shortcut)
    
        x_shortcut = Conv2D(filters3, (1, 1), strides=strides, 
                            padding='same', use_bias=False, kernel_initializer='he_normal', 
                           # activation = 'relu'
                           )(x_shortcut)
        
        #x_shortcut = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
         #                           channel_noise_factor=0.01)(x_shortcut)
        x_shortcut = BatchNormalization()(x_shortcut)
        
    
    
    # Add shortcut value to the main path
    x = Add()([x, x_shortcut])
    x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(x)
    
    x = ReLU()(x)
    
    #if use_attention:
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    #
    return x

def ResNet18(input_shape=(128, 128, 3), num_classes=2):
    input_tensor = Input(shape=input_shape)
    
    #attention_noise_output = GlobalAttentionNoiseLayer(noise_factor=0.2)(input_tensor)
    #print('attention_noise_output shape:', attention_noise_output.shape)
    x1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(input_tensor)
    
    #combine_input = Add()([attention_noise_output, x1])
    #print('combine_input shape:', combine_input.shape)

    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
    #                                channel_noise_factor=1.0)(input_tensor)
    x = x1
    x = Conv2D(64, (3, 3), strides=(2, 2), padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    x = BatchNormalization()(x)
    #x = ReLU()(x)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    x = MaxPooling2D((7,7), strides=(2, 2), padding='same')(x)
    
       
    # Residual blocks
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    
    
    x = residual_block(x, [64, 64, 256], strides=(1, 1))
    
    
    x = residual_block(x, [64, 64, 256])
    
    
    x = residual_block(x, [128, 128, 512])
    
    
    x = residual_block(x, [128, 128, 512])
        

    
    #x = residual_block(x, [256, 256, 1024])
    #x = residual_block(x, [256, 256, 1024])
    
    #x = residual_block(x, [512, 512, 2048])
    #x = residual_block(x, [512, 512, 2048])
    
    # Global average pooling
    #x = AveragePooling2D((2, 2))(x)
    x = Flatten()(x)
    #x = Flatten()(x)
    
    # Fully connected layer
    x = Dense(num_classes, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=input_tensor, outputs=x)
    
    return model

# Create ResNet-18 model
model = ResNet18()
#model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lr = 0.005

# Choose an optimizer and pass the learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Train the model
#model.fit(images_train, y_train_one_hot, epochs=10, batch_size=32, validation_split=0.2)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)
#model.summary()


In [ ]:
model.evaluate(images_test, y_test_one_hot)

In [ ]:
model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)

In [ ]:
best_model.evaluate(images_test, y_test_one_hot)
model.evaluate(images_test, y_test_one_hot)

In [ ]:
model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)

In [ ]:
from tensorflow.keras.models import load_model

# Load the best model saved during training

custom_objects = {'CombinedAttentionNoiseLayer': CombinedAttentionNoiseLayer
                  #, 
                 #'GlobalAttentionNoiseLayer': GlobalAttentionNoiseLayer
                 }

# Load the best model saved during training with custom layer
best_model = load_model('/working/best_models1.h5', custom_objects=custom_objects)

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, GlobalAveragePooling2D, ReLU, Add, AveragePooling2D, Flatten, Dense

import tensorflow as tf
from tensorflow.keras.layers import Input, MaxPooling2D, Conv2D, BatchNormalization, ReLU, Add, AveragePooling2D, Flatten, Dense

def residual_block(x, filters, strides=(1, 1), use_attention=True):
    filters1, filters2 = filters
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, channel_noise_factor=0.1)(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters2, (1, 1), strides=strides, padding='same', 
               , use_bias=False, kernel_initializer='he_normal')(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
     #                               channel_noise_factor=0.01)(x)
    
    #x = BatchNormalization()(x)
    #x = BatchNormalization()(x)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
   
    x_shortcut = x
    #print('x_shortcut shape:', x_shortcut.shape)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    # First block
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters1, (3, 3), strides=strides, padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    x = BatchNormalization()(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
           #                         channel_noise_factor=0.3)(x)
   
    # Second block
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    x = Conv2D(filters2, (3, 3), padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
     #                               channel_noise_factor=0.01)(x)
    
    x = BatchNormalization()(x)
    
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    
    # Third block
    
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.1, channel_noise_factor=0.1)(x)
    
    # Adjust shortcut connection dimensions to match main path
    if strides != (1, 1) or x.shape[-1] != filters2:
        #x_shortcut = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
         #                           channel_noise_factor=1.0)(x_shortcut)
        
        x_shortcut = Conv2D(filters2, (1, 1), strides=strides, 
                            padding='same', use_bias=False, kernel_initializer='he_normal', 
                           # activation = 'relu'
                           )(x_shortcut)
        
        #x_shortcut = CombinedAttentionNoiseLayer(spatial_noise_factor=0.01, 
         #                           channel_noise_factor=0.01)(x_shortcut)
        x_shortcut = BatchNormalization()(x_shortcut)
        
    
    
    # Add shortcut value to the main path
    x = Add()([x, x_shortcut])
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
     #                               channel_noise_factor=1.0)(x)
    
    x = ReLU()(x)
    
    #if use_attention:
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    #
    return x

def ResNet18(input_shape=(128, 128, 3), num_classes=2):
    input_tensor = Input(shape=input_shape)
    
    #attention_noise_output = GlobalAttentionNoiseLayer(noise_factor=0.2)(input_tensor)
    #print('attention_noise_output shape:', attention_noise_output.shape)
    '''x1 = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
                                    channel_noise_factor=1.0)(input_tensor)
    '''
    #combine_input = Add()([attention_noise_output, x1])
    #print('combine_input shape:', combine_input.shape)

    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=1.0, 
    #                                channel_noise_factor=1.0)(input_tensor)
    x = input_tensor
    x = Conv2D(64, (7, 7), strides=(2, 2), padding='same'
              , use_bias=False, kernel_initializer='he_normal', activation = 'relu')(x)
    
    x = BatchNormalization()(x)
    #x = ReLU()(x)
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
     #                               channel_noise_factor=0.3)(x)
   
    x = MaxPooling2D((7,7), strides=(2, 2), padding='same')(x)
    
       
    # Residual blocks
    #x = CombinedAttentionNoiseLayer(spatial_noise_factor=2.0, 
     #                               channel_noise_factor=2.0)(x)
    
    
    x = residual_block(x, [64, 64], strides=(1, 1))
    
    x = residual_block(x, [64, 64])
    x = residual_block(x, [64, 64])
    
    
    x = residual_block(x, [128, 128,], strides=(2, 2))
    x = residual_block(x, [128, 128])
    x = residual_block(x, [128, 128])
    x = residual_block(x, [128, 128])
    x = residual_block(x, [128, 128])
        

    
    #x = residual_block(x, [256, 256, 1024])
    #x = residual_block(x, [256, 256, 1024])
    
    #x = residual_block(x, [512, 512, 2048])
    #x = residual_block(x, [512, 512, 2048])
    
    # Global average pooling
    #x = AveragePooling2D((2, 2))(x)
    x = Flatten()(x)
    #x = Flatten()(x)
    
    # Fully connected layer
    x = Dense(num_classes, activation='sigmoid')(x)
    
    model = tf.keras.Model(inputs=input_tensor, outputs=x)
    
    return model

# Create ResNet-18 model
model = ResNet18()
#model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
lr = 0.005

# Choose an optimizer and pass the learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
# Train the model
#model.fit(images_train, y_train_one_hot, epochs=10, batch_size=32, validation_split=0.2)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=100, callbacks = callbacks,
          validation_split=0.2)
#model.summary()


In [ ]:
from tensorflow.keras.engine import get_source_inputs

In [ ]:
model.evaluate(images_test, y_test_one_hot)

In [ ]:
best_model.evaluate(images_test, y_test_one_hot)

In [ ]:
import numpy as np
model.evaluate(images_test, labels_test)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.1,
        nb_iter=20,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, labels_test)


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNet
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Load the pre-trained MobileNetV2 model with ImageNet weights
base_model = MobileNet(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the layers in the base model
for layer in base_model.layers:
    layer.trainable = False

# Define the custom head for fine-tuning
model = models.Sequential()
model.add(base_model)
model.add(layers.GlobalAveragePooling2D())
model.add(layers.Dense(256, activation='relu'))
model.add(layers.Dropout(0.5))
model.add(layers.Dense(2, activation='softmax'))  # Adjust for your specific classification task

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fine-tune the model on your specific dataset
model.fit(images_train, labels_train, epochs=5, batch_size=32, validation_split=0.2)



In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = projected_gradient_descent(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.25,
        nb_iter=20,
        norm=np.inf,
        loss_fn=None,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        rand_init=None,
        rand_minmax=None,
        sanity_checks=False,
        )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import numpy as np
best_model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=best_model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.1,
        nb_iter=20,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
best_model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = projected_gradient_descent(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.25,
        nb_iter=40,
        norm=np.inf,
        loss_fn=None,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        rand_init=None,
        rand_minmax=None,
        sanity_checks=False,
        )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import numpy as np
best_model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = projected_gradient_descent(
        model_fn=best_model,
        x=images_test[start_idx:end_idx],
        eps=32/255,
        eps_iter=0.031,
        nb_iter=100,
        norm=np.inf,
        loss_fn=None,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        rand_init=None,
        rand_minmax=None,
        sanity_checks=False,
        )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
best_model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
sca_best_model = best_model
sca_model = model

In [ ]:
sca_best_model1 = best_model ## 81
sca_model1 = model ## 74

In [ ]:
import numpy as np
best_model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=best_model,
        x=images_test[start_idx:end_idx],
        eps=0.125,
        eps_iter=0.0125,
        nb_iter=100,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
best_model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=0.125,
        eps_iter=0.0125,
        nb_iter=100,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)


In [ ]:
model_sca = model
model_best_sca = best_model

In [ ]:
model_mb = model

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model_mb,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model_mb.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=10,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=4,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=4,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.001,
                  0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09,0.1,
                  0.2]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.001,
                  0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09,0.1,
                  0.2]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=4,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,
                  
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=4,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.001,
                  0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,
                  0.2, 0.3]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.001,
                  0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1,
                  0.2, 0.3]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=10,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=4,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,
                  0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model1,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=4,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model1.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.3]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=best_model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(best_model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2, 0.3]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.1,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.1,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
y_train_one_hot.shape

In [ ]:
##gatn-ur
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
import tensorflow as tf


from tensorflow.keras.layers import Layer, Conv2D, InputSpec

class GlobalAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, noise_factor=0.2, **kwargs):
        super(GlobalAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor

    def build(self, input_shape):
        self.alpha = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='alpha')
        super(GlobalAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs), mean=0, stddev=self.noise_factor)
            return inputs + tf.multiply(self.alpha, noise)
        else:
            return inputs
        
class FeatureAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, noise_factor=0.2, learning_rate=0.001, **kwargs):
        super(FeatureAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor
        self.learning_rate = learning_rate

    def build(self, input_shape):
        _, H, W, C = input_shape
        self.alpha = self.add_weight(shape=(H, W, C), initializer='ones', trainable=True, name='alpha')
        
        self.dense = tf.keras.layers.Dense(units=C, activation='relu')
        super(FeatureAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs), mean=0, stddev=self.noise_factor)
            
            global_avg_pooled = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
            delta_related = self.dense(global_avg_pooled)
            
            # Calculate attention noise
            #attention_noise_related = (inputs + delta_related) * self.alpha * noise
            attention_noise = inputs + delta_related * inputs * self.alpha * noise
            self.add_loss(tf.reduce_mean(attention_noise))
            return attention_noise
        else:
            return inputs

    def get_config(self):
        config = super(FeatureAttentionNoiseLayer, self).get_config()
        config.update({'noise_factor': self.noise_factor, 'learning_rate': self.learning_rate})
        return config
    
num_classes = 4  # Adjust based on the number of classes in your dataset

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Flatten, Dense, Add, Concatenate
from tensorflow.keras.models import Model

# Load pre-trained VGG16 model (excluding top layers)
input_shape = (128, 128, 3)
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
# Apply combined attention noise to input data
#attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
 #                                                    channel_noise_factor=0.3)(input_data)
#attention_noise_output = GlobalAttentionNoiseLayer(spatial_noise_factor=0.3)(input_data)
attention_noise_output = GlobalAttentionNoiseLayer(noise_factor=0.1)(input_data)
print('attention_noise_output shape:', attention_noise_output.shape)
#print('attention_noise_output shape:', attention_noise_output.shape)
combine_input = Add()([attention_noise_output, input_data])
print('combine_input shape:', combine_input.shape)
vgg16_output = vgg16_base(combine_input)
print('vgg16_output shape:', vgg16_output.shape)
# Apply spatial attention noise to specific convolutional layers (e.g., the 6th, 12th, and 18th convolutional layers)

conv_layer_indices = [3, 6, 10, 14, 18]  # Indices of the convolutional layers in VGG16
x = attention_noise_output

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

for i, layer in enumerate(vgg16_base.layers):
    x = layer(x)
    #x = layer(x, name='inp')
    if i in conv_layer_indices:
        #x = SpatialAttentionLayer()(x)
        x = FeatureAttentionNoiseLayer(noise_factor=0.1)(x)
        


print('x shape:', x.shape)
# Add the output of VGG16 with spatial attention noise and the output of combined attention noise

#combined_output = tf.concat([x, vgg16_output], axis=-1)
#print('combined_output shape:', combined_output.shape)
# Add additional layers for classification
flatten_layer = Flatten()(x)
dense_layer = Dense(64, activation='selu')(flatten_layer)
output_layer = Dense(3, activation='softmax')(dense_layer)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()



In [ ]:

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=500, callbacks = callbacks,
          validation_split=0.2)

In [ ]:
model.evaluate(images_test, y_test_one_hot, batch_size=128)

In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.1,
        nb_iter=20,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot, batch_size=128)
from sklearn.metrics import roc_auc_score
y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
y_pred = y_pred >= 0.5
y_pred = np.array(y_pred, dtype='int32')
accuracy = accuracy_score(y_pred, y_test_one_hot) * 100
precision = precision_score(y_pred, y_test_one_hot, average='macro') * 100
recall = recall_score(y_pred, y_test_one_hot, average='macro') * 100
f1 = f1_score(y_pred, y_test_one_hot, average='macro') * 100
auc = roc_auc_score(y_pred, y_test_one_hot) * 100

#print('Epsilon value:', epsilon)
#print('Alpha value:', alpha)  # Print alpha value for reference
print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 Score:', f1)
print('auc Score:', auc)


In [ ]:
model.evaluate(images_test, y_test_one_hot)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.1,
                  
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=20,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.1,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples, batch_size=128))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
##gatn-r
import tensorflow as tf
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.layers import Input, Flatten, Dense
from tensorflow.keras.models import Model
import tensorflow as tf


from tensorflow.keras.layers import Layer, Conv2D, InputSpec

class GlobalAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, noise_factor=0.2, **kwargs):
        super(GlobalAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor

    def build(self, input_shape):
        self.alpha = self.add_weight(shape=(1,), initializer='ones', trainable=True, name='alpha')
        super(GlobalAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs), mean=0, stddev=self.noise_factor)
            return inputs + tf.multiply(self.alpha, noise)
        else:
            return inputs
        
class FeatureAttentionNoiseLayer(tf.keras.layers.Layer):
    def __init__(self, noise_factor=0.2, learning_rate=0.001, **kwargs):
        super(FeatureAttentionNoiseLayer, self).__init__(**kwargs)
        self.noise_factor = noise_factor
        self.learning_rate = learning_rate

    def build(self, input_shape):
        _, H, W, C = input_shape
        self.alpha = self.add_weight(shape=(H, W, C), initializer='ones', trainable=True, name='alpha')
        
        self.dense = tf.keras.layers.Dense(units=C, activation='relu')
        super(FeatureAttentionNoiseLayer, self).build(input_shape)

    def call(self, inputs, training=None):
        if training:
            noise = tf.random.normal(shape=tf.shape(inputs), mean=0, stddev=self.noise_factor)
            
            global_avg_pooled = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
            delta_related = self.dense(global_avg_pooled)
            
            # Calculate attention noise
            #attention_noise_related = (inputs + delta_related) * self.alpha * noise
            attention_noise = inputs + delta_related * self.alpha * noise
            self.add_loss(tf.reduce_mean(attention_noise))
            return attention_noise
        else:
            return inputs

    def get_config(self):
        config = super(FeatureAttentionNoiseLayer, self).get_config()
        config.update({'noise_factor': self.noise_factor, 'learning_rate': self.learning_rate})
        return config
    
num_classes = 4  # Adjust based on the number of classes in your dataset

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Input, Flatten, Dense, Add, Concatenate
from tensorflow.keras.models import Model

# Load pre-trained VGG16 model (excluding top layers)
input_shape = (128, 128, 3)
vgg16_base = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

# Freeze the layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

# Create custom attention noise layers
input_data = Input(shape=input_shape, name='input_data')
# Apply combined attention noise to input data
#attention_noise_output = CombinedAttentionNoiseLayer(spatial_noise_factor=0.3, 
 #                                                    channel_noise_factor=0.3)(input_data)
#attention_noise_output = GlobalAttentionNoiseLayer(spatial_noise_factor=0.3)(input_data)
attention_noise_output = GlobalAttentionNoiseLayer(noise_factor=0.2)(input_data)
print('attention_noise_output shape:', attention_noise_output.shape)
#print('attention_noise_output shape:', attention_noise_output.shape)
combine_input = Add()([attention_noise_output, input_data])
print('combine_input shape:', combine_input.shape)
vgg16_output = vgg16_base(combine_input)
print('vgg16_output shape:', vgg16_output.shape)
# Apply spatial attention noise to specific convolutional layers (e.g., the 6th, 12th, and 18th convolutional layers)

conv_layer_indices = [3, 6, 10, 14, 18]  # Indices of the convolutional layers in VGG16
x = attention_noise_output

import tensorflow as tf
from tensorflow.keras.layers import Layer, Conv2D

for i, layer in enumerate(vgg16_base.layers):
    x = layer(x)
    #x = layer(x, name='inp')
    if i in conv_layer_indices:
        #x = SpatialAttentionLayer()(x)
        x = FeatureAttentionNoiseLayer(noise_factor=0.2)(x)
        


print('x shape:', x.shape)
# Add the output of VGG16 with spatial attention noise and the output of combined attention noise

#combined_output = tf.concat([x, vgg16_output], axis=-1)
#print('combined_output shape:', combined_output.shape)
# Add additional layers for classification
flatten_layer = Flatten()(x)
dense_layer = Dense(128, activation='selu')(flatten_layer)
output_layer = Dense(3, activation='softmax')(dense_layer)

# Create the final model
model = Model(inputs=input_data, outputs=output_layer)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Print the model summary
#model.summary()

# Continue with training the model...
# Continue with training the model...

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
def checkpoint_callback():

    checkpoint_filepath = 'best_models1.h5'

    model_checkpoint_callback= ModelCheckpoint(filepath=checkpoint_filepath,
                           save_weights_only=False,
                           frequency='epoch',
                           monitor='val_loss',
                           save_best_only=True,
                           verbose=1)

    return model_checkpoint_callback

def early_stopping(patience):
    es_callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, verbose=1)
    return es_callback



checkpoint_callback = checkpoint_callback()
early_stopping = early_stopping(patience=100)
callbacks = [checkpoint_callback, early_stopping]

model.fit(images_train, y_train_one_hot, epochs=500, callbacks = callbacks,
          validation_split=0.2)


In [ ]:
import numpy as np
model.evaluate(images_test, y_test_one_hot)
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Split the evaluation into batches
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

adversarial_examples = []

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

    # Generate adversarial examples for the current batch
    adv_batch = momentum_iterative_method(
        model_fn=model,
        x=images_test[start_idx:end_idx],
        eps=1.0,
        eps_iter=0.1,
        nb_iter=20,
        norm=np.inf,
        clip_min=None,
        clip_max=None,
        y=labels_test[start_idx:end_idx],
        targeted=False,
        decay_factor=1.0,
        sanity_checks=True,
    )

    adversarial_examples.append(adv_batch)

# Concatenate the adversarial examples from all batches
adversarial_examples = np.concatenate(adversarial_examples, axis=0)

# Evaluate the model on the concatenated adversarial examples
model.evaluate(adversarial_examples, y_test_one_hot)
from sklearn.metrics import roc_auc_score
y_pred = tf.squeeze(model.predict(adversarial_examples))
y_pred = y_pred >= 0.5
y_pred = np.array(y_pred, dtype='int32')
accuracy = accuracy_score(y_pred, y_test_one_hot) * 100
precision = precision_score(y_pred, y_test_one_hot, average='macro') * 100
recall = recall_score(y_pred, y_test_one_hot, average='macro') * 100
f1 = f1_score(y_pred, y_test_one_hot, average='macro') * 100
auc = roc_auc_score(y_pred, y_test_one_hot) * 100

#print('Epsilon value:', epsilon)
#print('Alpha value:', alpha)  # Print alpha value for reference
print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1 Score:', f1)
print('auc Score:', auc)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.1,
                  
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=20,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.momentum_iterative_method import momentum_iterative_method

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 1,
                  0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=20,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.projected_gradient_descent import projected_gradient_descent
# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  projected_gradient_descent(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/4,
            nb_iter=100,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=None,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.basic_iterative_method import basic_iterative_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

#adversarial_examples = []

# Define epsilon values
epsilon_values = [0, 0.1,0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  basic_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            rand_init=None,
            rand_minmax=0.3,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Choose a batch size
batch_size = 20

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from cleverhans.tf2.attacks.fast_gradient_method import fast_gradient_method
# Choose a batch size
batch_size = 20
num_samples = len(images_test)
num_batches = (num_samples + batch_size - 1) // batch_size

# Define epsilon values
epsilon_values = [0, 0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,
                  0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19]

# List to store results for each epsilon
results_per_epsilon = []

for epsilon in epsilon_values:
    adversarial_examples = []

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = (i + 1) * batch_size if (i + 1) * batch_size < num_samples else num_samples

        # Generate adversarial examples for the current batch
        '''
        adv_batch = momentum_iterative_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            eps_iter=epsilon/10,
            nb_iter=100,
            norm=np.inf,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            decay_factor=1.0,
            sanity_checks=True,
        )
        '''
        adv_batch =  fast_gradient_method(
            model_fn=model,
            x=images_test[start_idx:end_idx],
            eps=epsilon,
            norm=np.inf,
            loss_fn=None,
            clip_min=None,
            clip_max=None,
            y=labels_test[start_idx:end_idx],
            targeted=False,
            sanity_checks=False,
        )

        adversarial_examples.append(adv_batch)

    # Concatenate the adversarial examples from all batches
    adversarial_examples = np.concatenate(adversarial_examples, axis=0)

    # Evaluate the model on the concatenated adversarial examples
    y_pred = tf.squeeze(model.predict(adversarial_examples))
    y_pred_binary = y_pred >= 0.5
    y_pred_binary = np.array(y_pred_binary, dtype='int32')

    # Calculate evaluation metrics for the current epsilon
    accuracy = accuracy_score(y_pred_binary, y_test_one_hot) * 100
    precision = precision_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    recall = recall_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    f1 = f1_score(y_pred_binary, y_test_one_hot, average='macro') * 100
    #auc = roc_auc_score(y_pred, y_test_one_hot) * 100

    # Store results for the current epsilon
    results_per_epsilon.append({
        'epsilon': epsilon,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        #'auc': auc
    })

# Print or use the results as needed
for result in results_per_epsilon:
    print(f"Epsilon: {result['epsilon']}")
    print(f"Accuracy: {result['accuracy']}")
    print(f"Precision: {result['precision']}")
    print(f"Recall: {result['recall']}")
    print(f"F1 Score: {result['f1']}")
    #print(f"AUC Score: {result['auc']}")
    print('-' * 50)
